# Setup

In [1]:
import warnings
warnings.filterwarnings("ignore")
warnings.simplefilter(action="ignore", category=FutureWarning)

In [2]:
from utils import setup
from indicators import  compute_indicators

In [3]:
from models import SparseResNet18
#surrogate model
surrogate_model = None

In [4]:
from pathlib import Path
import torch
from robustbench.utils import download_gdrive
from secmlt.models.pytorch.base_pytorch_nn import BasePytorchClassifier

device = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_ID = "1Af_owmMvg1LxjITLE1gFUmPx5idogeTP"
gamma = 0.1

pretrained_path = Path("models")
pretrained_path.mkdir(parents=True, exist_ok=True)

filepath = pretrained_path / f"kwta_spresnet18_{gamma}_cifar_adv.pth"
if not filepath.exists():
    download_gdrive(MODEL_ID, filepath)

model_net = SparseResNet18(sparsities=[gamma, gamma, gamma, gamma])
state_dict = torch.load(filepath, map_location=device)
model_net.load_state_dict(state_dict)
model_net.eval()
model_net.to(device)

wrapped_model = BasePytorchClassifier(model_net)

Download started: path=models\kwta_spresnet18_0.1_cifar_adv.pth (gdrive_id=1Af_owmMvg1LxjITLE1gFUmPx5idogeTP)
Download finished: path=models\kwta_spresnet18_0.1_cifar_adv.pth (gdrive_id=1Af_owmMvg1LxjITLE1gFUmPx5idogeTP)


# IoAF

In [5]:
attack, model, dataloader = setup(
    num_samples=3,
    n_batches=1,
    num_steps=20,
    step_size=0.005,
    epsilon=8/255,
    model=wrapped_model,
    dataset_name="cifar10",
    threat_model=None,
    model_class=None,
    model_id=None,
    model_path=None,
    subset_size=3,
)

ds = compute_indicators(attack, model, dataloader, surrogate_model)
ds

starting pre-processing...
end pre-processing

starting unavailable_gradients_indicator...
end unavailable_gradients_indicator

staring unstable_predictions_indicator...
end unstable_predictions_indicator

staring silent_success_indicator...
end silent_success_indicator

starting incomplete_optimization_indicator...
end incomplete_optimization_indicator

no surrogate

starting unconstrained_attack_failure_indicator..
end unconstrained_attack_failure_indicator

starting Attack_fails...
end Attack_fails



,attack_fails,unavailable_gradients,unstable_predictions,silent_success,incomplete_optimization,transfer_failure,unconstrained_attack_failure
0,0.666667,False,0.978674,False,0.666667,None,0.333333


# Unconstrained PGD PredictionTracker

In [6]:
attack.trackers[0].trackers[1].get()

tensor([[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 3, 2, 3, 3, 3, 3],
        [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8],
        [8, 8, 8, 8, 8, 8, 8, 8, 0, 0, 0, 8, 8, 8, 8, 8, 8, 1, 1, 8]])